# Определение ориентации текста: 0° vs 180°

Цель — для каждой тестовой картинки предсказать вероятность `p_180`, что текст на изображении перевёрнут на 180°.

Основная метрика — `1 - Brier Score`, поэтому модель должна выдавать корректно откалиброванную вероятность, а не только бинарный класс.

В решении используются только открытые сторонние OCR-датасеты.
Ручная разметка тестовых данных и внешние API не используются.

## Краткое описание решения

Для обучения использовались открытые OCR-датасеты GOST, ICDAR2015 и CTW1500.

Исходные изображения были случайно разделены на train и validation
в пропорции 80/20 с фиксированным `seed=42`. Разбиение выполнялось до
генерации повернутых копий, чтобы исключить попадание одного и того же
изображения в train и validation в разных ориентациях.

Validation-выборка не использовалась для обучения и применялась для оценки
обобщающей способности модели на ранее не виденных изображениях. После каждой
эпохи на validation рассчитывались `1 - Brier Score` и accuracy. Основной
метрикой для выбора лучшего checkpoint был `1 - Brier Score`, поскольку именно
эта метрика используется в задаче. По validation-метрике также выполнялся
early stopping.

Для каждого изображения создавались два примера: исходная ориентация
(`label=0`) и поворот на 180° (`label=1`).

Использовалась компактная CNN. Модель обучалась с `CrossEntropyLoss`.

Модель состоит из последовательности сверточных блоков с постепенным увеличением
числа каналов `16 → 32 → 64 → 128`. Каждый блок использует свёртку,
Batch Normalization и ReLU, а на ранних уровнях дополнительно применяются
Max Pooling и Dropout.

После извлечения признаков используется `Adaptive Average Pooling`, который
преобразует карту признаков в 128-мерный вектор. Финальный линейный слой
выдаёт logits для двух классов: исходная ориентация (`0°`) и поворот на `180°`.

Архитектура намеренно сделана компактной, поскольку в задаче помимо качества
учитываются скорость и размер модели.

На этапе инференса для каждого тестового изображения берётся вероятность
класса `1`, которая сохраняется в `submission.csv` как `p_180`.

Для разбиения данных и их перемешивания используется фиксированный seed=42. Перед созданием модели также вызывается torch.manual_seed(42). Эти настройки уменьшают случайную вариативность обучения, но не гарантируют полного совпадения результатов на разных устройствах.
Для воспроизведения итоговых предсказаний предусмотрен отдельный блокнот predict.ipynb, который загружает сохранённые веса и выполняет предсказание без повторного обучения. Исходный CSV получен на MPS; при использовании другого устройства возможны небольшие численные различия в вероятностях.

## Проведённые эксперименты
Помимо итоговой CNN с шестью свёрточными слоями, были рассмотрены два варианта:
- MobileNetV3-Small с предобученными весами ImageNet. Последний классификационный слой был заменён на слой с двумя выходами, после чего дообучались все слои модели на CTW1500. В сохранённом логе первых восьми эпох максимальное значение 1 − Brier Score составило примерно 0.7690. Этот вариант не был выбран для итогового решения. При этом эксперимент отличался от финального составом данных, предобработкой и настройками обучения, поэтому результаты нельзя считать прямым сравнением только архитектур.
- CNN с семью свёрточными слоями. Проверялось, улучшит ли увеличение глубины качество определения ориентации текста. В проведённом эксперименте эта модель показала более низкий результат, чем вариант с шестью свёрточными слоями, поэтому дополнительный слой не вошёл в итоговую архитектуру.Её лучший 1 − Brier Score на используемой валидационной выборке составил 0.8861.

Для финального решения выбрана CNN с шестью свёрточными слоями и 393 858 параметрами. Её лучший 1 − Brier Score на используемой валидационной выборке составил 0.9123.

In [9]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

## Данные

Для обучения использовались открытые OCR-датасеты:

- **GOST** — выбран как источник текстовых изображений с печатными символами и документоподобным оформлением;
- **ICDAR2015 OCR** — добавлен для увеличения разнообразия условий съёмки и качества текста, включая более сложные реальные изображения;
- **CTW1500** — использован для расширения вариативности форм, расположения и визуального представления текста.

GOST, ICDAR2015 OCR и CTW1500 были выбраны как взаимодополняющие OCR-датасеты
с различающимся визуальным представлением текста. 
Совместное использование нескольких датасетов позволяет увеличить разнообразие
обучающих примеров и снизить зависимость модели от особенностей одного конкретного
источника данных.

Из каждого датасета используются изображения с текстом.
После объединения данных исходные изображения разбиваются на train и validation
в пропорции 80/20 с фиксированным `seed=42`.

Разбиение выполняется до создания повернутых копий изображения.
Это важно, чтобы оригинал и его версия, повернутая на 180°, не могли попасть
в разные части train/validation и вызвать утечку данных.

In [10]:
from datasets import load_dataset

ds_gost = load_dataset(
    "Mkz-Prog/gost-ru-technical-ocr-dataset",
    split="train"
)

ds_gost = ds_gost.select_columns(["image"])

print(f"GOST: {len(ds_gost)} изображений")

Resolving data files:   0%|          | 0/1001 [00:00<?, ?it/s]

GOST: 1000 изображений


In [14]:
ds_icdar_train = load_dataset(
    "MiXaiLL76/ICDAR2015_OCR",
    split="train"
)

ds_icdar_train = ds_icdar_train.select_columns(["image"])

print(f"ICDAR train: {len(ds_icdar_train)}")

ICDAR train: 4468


In [11]:
# Это test-часть стороннего OCR-датасета.
# Используем её как источник изображений для собственного разбиения.
ds_icdar_test = load_dataset(
    "MiXaiLL76/ICDAR2015_OCR",
    split="test"
)

ds_icdar_test = ds_icdar_test.select_columns(["image"])

print(f"ICDAR test: {len(ds_icdar_test)}")

ICDAR test: 2077


In [ ]:
ds_ctw1500 = load_dataset(
    "MiXaiLL76/CTW1500_OCR",
    split="train"
)

print(f"Исходный CTW1500: {len(ds_ctw1500)}")

# Берём фиксированную случайную подвыборку из 3000 изображений.
# seed обеспечивает повторяемость выбора при неизменном исходном датасете.
ds_ctw1500 = (
    ds_ctw1500
    .shuffle(seed=42)
    .select(range(3000))
    .select_columns(["image"])
)

print(f"CTW1500 после выборки: {len(ds_ctw1500)}")

Исходный CTW1500: 7703
CTW1500 после выборки: 3000


In [16]:
from datasets import concatenate_datasets

ds = concatenate_datasets([
    ds_gost,
    ds_icdar_train,
    ds_icdar_test,
    ds_ctw1500
])

print(f"Всего изображений: {len(ds)}")

Всего изображений: 10545


In [1]:
import copy
import csv
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import concatenate_datasets
from PIL import Image

In [2]:
SEED = 42
IMAGE_WIDTH, IMAGE_HEIGHT = 384, 64
BATCH_SIZE = 256
torch.manual_seed(SEED)

## Формирование целевой переменной

Для каждого исходного изображения создаются два примера:

- исходное изображение — `label = 0`;
- изображение, повернутое на 180° — `label = 1`.

In [17]:
# Предполагаем, что исходные изображения ориентированы правильно.
# Исходному изображению присваиваем 0, повёрнутому на 180° — 1.
def make_orientation_dataset(source_ds, seed=SEED):
    upright = source_ds.map(lambda _: {"label": 0})
    rotated = source_ds.map(
        lambda sample: {
            "image": sample["image"].transpose(Image.Transpose.ROTATE_180),
            "label": 1,
        }
    )
    return concatenate_datasets([upright, rotated]).shuffle(seed=seed)

In [18]:
# Сначала делим исходные изображения, затем создаём обе ориентации.
# Так оригинал и его поворот не попадут одновременно в train и validation.
base_splits = ds.train_test_split(test_size=0.20, seed=SEED, shuffle=True)

train_hf = make_orientation_dataset(base_splits["train"])
val_hf = make_orientation_dataset(base_splits["test"])

print(f"Исходные: train={len(base_splits['train'])}, test={len(base_splits['test'])}")
print(f"После поворота: train={len(train_hf)}, validation={len(val_hf)}")

Исходные: train=8436, test=2109
После поворота: train=16872, validation=4218


In [3]:
import torchvision.transforms as T

In [19]:
class OrientationDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        image = sample["image"].convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        pixels = np.asarray(image, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(pixels).permute(2, 0, 1)
        return tensor, torch.tensor(sample["label"], dtype=torch.long)

## Предобработка и аугментации

Все изображения приводятся к размеру `384 × 64`.

Используются следующие преобразования:

- **Grayscale** — цвет в данной задаче не является существенным признаком для
  определения ориентации текста. Перевод в оттенки серого уменьшает зависимость
  модели от цветовой палитры и заставляет её в большей степени ориентироваться на форму символов и структуру текста.

- **Изменение яркости и контраста** — имитирует различия между сканами,
  фотографиями и изображениями из разных источников: неравномерное освещение,
  блеклый текст, высокий или низкий контраст. Это снижает зависимость модели
  от конкретного качества изображения.

- **Случайная инверсия** — позволяет модели одинаково хорошо работать как
  с тёмным текстом на светлом фоне, так и со светлым текстом на тёмном фоне.
  Благодаря этому знак контраста не становится признаком конкретного класса.

- **Небольшие affine-преобразования** — моделируют небольшие отклонения,
  возникающие при сканировании или фотографировании текста: небольшой наклон
  и изменение масштаба. Угол преобразования специально ограничен несколькими
  градусами, поскольку сильные повороты могли бы изменить смысл целевой задачи
  `0° vs 180°`.

- **Gaussian blur** — имитирует небольшую расфокусировку, низкое качество
  сканирования и потерю резкости. Это помогает модели не слишком сильно
  зависеть от мелких высокочастотных деталей изображения.

Аугментации применяются только к train-выборке. Для validation и test
используется детерминированная предобработка, чтобы оценка качества не зависела
от случайно применённых преобразований. 

Основная цель аугментаций — сделать модель менее чувствительной к источнику,
качеству и оформлению изображения, сохранив при этом признак, который необходимо
предсказывать: ориентацию текста.

In [20]:
train_transforms = T.Compose([
    # Оставляем три одинаковых канала серого, поскольку модель принимает 3 канала.
    T.Grayscale(num_output_channels=3), 
    T.Resize(
    (IMAGE_HEIGHT, IMAGE_WIDTH),
    interpolation=Image.Resampling.BILINEAR,
    ),
    T.RandomInvert(p=0.5),
    T.ColorJitter(brightness=(0.2, 1.5), contrast=(0.2, 1.5)),
    T.RandomAffine(degrees=3, scale=(0.8, 1.2), fill=255),
    T.RandomApply([T.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0))], p=0.3)
])

val_transforms = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize((IMAGE_HEIGHT, IMAGE_WIDTH), interpolation=Image.Resampling.BILINEAR)
])

train_loader = DataLoader(
    OrientationDataset(train_hf, transform=train_transforms), 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0
)

val_loader = DataLoader(
    OrientationDataset(val_hf, transform=val_transforms), 
    batch_size=128, 
    shuffle=False, 
    num_workers=0
)

## Модель

Используется компактная сверточная нейронная сеть.

Она последовательно извлекает визуальные признаки текста с помощью Conv2d,
BatchNorm, ReLU и MaxPool, после чего выполняется глобальная агрегация признаков
и бинарная классификация.

Модель намеренно сделана небольшой, чтобы соблюдать баланас между скоростью и компактностью решения.

## Архитектура CNN

Структура модели:

`Input (3 × H × W)`

→ `Conv 3→16 + BatchNorm + ReLU + MaxPool + Dropout`

→ `Conv 16→32 + BatchNorm + ReLU + MaxPool + Dropout`

→ `Conv 32→64 + BatchNorm + ReLU + MaxPool + Dropout`

→ `Conv 64→128 + BatchNorm + ReLU`

→ `Conv 128→128 + BatchNorm + ReLU`

→ `Conv 128→128 + BatchNorm + ReLU`

→ `AdaptiveAvgPool(1×1)`

→ `Dropout`

→ `Linear(128 → 2)`

Первые три блока уменьшают пространственное разрешение изображения и постепенно
увеличивают число признаковых каналов. Последующие свёртки извлекают более
высокоуровневые признаки текста.

Dropout добавлен в первые три блока как регуляризация ранних и промежуточных
признаков. В более глубоких слоях он не используется, чтобы сохранить устойчивые
высокоуровневые признаки ориентации текста. Перед финальным линейным слоем
дополнительно применяется `Dropout(0.30)`.

`AdaptiveAvgPool2d((1, 1))` агрегирует пространственную информацию каждого
из 128 каналов, после чего классификатор выдаёт два logits.
Вероятность класса `1` после softmax интерпретируется как `p_180`.

In [10]:
class TinyOrientationNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.10),
            
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout2d(0.20),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        
        self.classifier = nn.Sequential(nn.Dropout(0.30), nn.Linear(128, 2))

    def forward(self, images):
        return self.classifier(self.features(images).flatten(1))

## Обучение и валидация

Для обучения используется `CrossEntropyLoss`.

После каждой эпохи модель оценивается на validation-выборке.
Основная validation-метрика соответствует метрике соревнования:

`score = 1 - Brier Score`

Дополнительно считается accuracy только как вспомогательная метрика.

Лучший checkpoint выбирается по максимальному значению `1 - Brier Score`.
Также используется early stopping, чтобы предотвратить переобучение модели.

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = TinyOrientationNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
print(f"Устройство: {device}")

Устройство: mps


In [ ]:
def validate(model, loader):
    model.eval()
    probabilities, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            # Класс 1 соответствует перевороту на 180°.
            probabilities.append(torch.softmax(logits, dim=1)[:, 1])
            targets.append(labels)
    probabilities = torch.cat(probabilities).cpu().numpy()
    targets = torch.cat(targets).cpu().numpy()
    # Brier Score — средняя квадратичная ошибка вероятностей.
    # Максимизируем 1 - Brier Score.
    brier_score = np.mean((probabilities - targets) ** 2)
    accuracy = np.mean((probabilities >= 0.5) == targets)
    return 1 - brier_score, accuracy

In [ ]:
MAX_EPOCHS, PATIENCE = 300, 10
best_score, best_state, stale_epochs = -np.inf, None, 0

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = torch.zeros((), device=device)
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.detach() * len(labels)

    epoch_loss = total_loss.item() / len(train_hf)
    score, accuracy = validate(model, val_loader)
    print(f"epoch {epoch + 1}: loss={epoch_loss:.4f}, accuracy={accuracy:.4f}, 1 - Brier Score={score:.4f}")
    # Считаем улучшением прирост метрики больше 1e-5.
    if score > best_score + 1e-5:
        # Создаём независимую копию лучших весов:
        # дальнейшее обучение не должно изменять сохранённое состояние.
        best_score, best_state, stale_epochs = score, copy.deepcopy(model.state_dict()), 0
    else:
        stale_epochs += 1
        # После PATIENCE эпох без такого улучшения останавливаем обучение.
        if stale_epochs >= PATIENCE:
            print(f"Early stopping: нет улучшения {PATIENCE} эпох.")
            break
# Возвращаем веса лучшей эпохи по validation, а не последней эпохи.
model.load_state_dict(best_state)
print(f"Лучший 1 - Brier Score: {best_score:.4f}")

epoch 1: loss=0.7040, accuracy=0.5014, 1 - Brier Score=0.7498
epoch 2: loss=0.6986, accuracy=0.5002, 1 - Brier Score=0.7492
epoch 3: loss=0.6959, accuracy=0.5057, 1 - Brier Score=0.7500
epoch 4: loss=0.6957, accuracy=0.5026, 1 - Brier Score=0.7479
epoch 5: loss=0.6941, accuracy=0.4998, 1 - Brier Score=0.7498
epoch 6: loss=0.6944, accuracy=0.5040, 1 - Brier Score=0.7499
epoch 7: loss=0.6937, accuracy=0.5083, 1 - Brier Score=0.7501
epoch 8: loss=0.6928, accuracy=0.5130, 1 - Brier Score=0.7511
epoch 9: loss=0.6934, accuracy=0.5138, 1 - Brier Score=0.7507
epoch 10: loss=0.6906, accuracy=0.5735, 1 - Brier Score=0.7541
epoch 11: loss=0.6866, accuracy=0.5346, 1 - Brier Score=0.7510
epoch 12: loss=0.6790, accuracy=0.5344, 1 - Brier Score=0.7514
epoch 13: loss=0.6739, accuracy=0.5605, 1 - Brier Score=0.7592
epoch 14: loss=0.6686, accuracy=0.6249, 1 - Brier Score=0.7758
epoch 15: loss=0.6507, accuracy=0.5989, 1 - Brier Score=0.7692
epoch 16: loss=0.6436, accuracy=0.6337, 1 - Brier Score=0.7854
e

In [ ]:
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "best_orientation_model_7conv.pt"

torch.save({"model_state_dict": model.state_dict(), "best_validation_score": float(best_score)}, CHECKPOINT_PATH)
print(f"Лучшие веса сохранены: {CHECKPOINT_PATH}")

Лучшие веса сохранены: artifacts/best_orientation_model_7conv.pt


## Результаты валидации

Лучший validation score:

`1 - Brier Score = 0.9123`

Модель с лучшим validation score сохраняется и используется для инференса
на тестовых изображениях.

Ограничение текущей схемы валидации: train и validation содержат изображения
из одних и тех же внешних источников данных. Поэтому validation хорошо измеряет
качество на отложенных примерах этих датасетов, но может хуже отработать на тестовом примере.

In [4]:
TEST_IMAGES_DIR = Path("test/test/images")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not TEST_IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Не найдена папка {TEST_IMAGES_DIR}. Поместите туда тестовые изображения.")
image_paths = sorted(path for path in TEST_IMAGES_DIR.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
if not image_paths:
    raise FileNotFoundError(f"В папке {TEST_IMAGES_DIR} нет изображений.")
print(f"Тестовых изображений: {len(image_paths)}")

Тестовых изображений: 20000


In [12]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Лучшие веса загружены.")

Лучшие веса загружены.


In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as opened_image:
            image = val_transforms(opened_image.convert("RGB"))
        pixels = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(pixels).permute(2, 0, 1), path.stem

In [ ]:
test_loader = DataLoader(TestImageDataset(image_paths), batch_size=128, shuffle=False, num_workers=0)
rows = []
with torch.no_grad():
    for images, image_ids in test_loader:
        p_180 = torch.softmax(model(images.to(device)), dim=1)[:, 1].cpu().numpy()
        rows.extend(zip(image_ids, p_180))
print(f"Предсказано изображений: {len(rows)}")

Предсказано изображений: 20000


In [ ]:
SUBMISSION_PATH = Path("submission.csv")
with SUBMISSION_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["image_id", "p_180"])
    writer.writerows((image_id, f"{p_180:.8f}") for image_id, p_180 in rows)

print(f"Файл сохранён: {SUBMISSION_PATH}")